## Load the webdocs

In [ ]:
!pip install -qU langchain_community beautifulsoup4

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

FILE_PATHS = [
    "https://docs.langchain.com/oss/python/integrations/document_loaders",
    "https://docs.langchain.com/oss/python/integrations/vectorstores",
    "https://docs.langchain.com/oss/python/integrations/text_embedding",
]

loader = WebBaseLoader(web_path=FILE_PATHS)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(documents[0].page_content[:500])

Loaded 3 documents
Document loader integrations - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationIntegrations by componentDocument loader integratio


##Text Spliter

In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 200,
)
all_splits = text_splitter.split_documents(documents)
print(all_splits)
print(len(all_splits))

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/integrations/document_loaders', 'title': 'Document loader integrations - Docs by LangChain', 'description': 'Integrate with document loaders using LangChain Python.', 'language': 'en'}, page_content="Document loader integrations - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationIntegrations by componentDocument loader integrationsOverviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLangChain integrationsAll providersPopular ProvidersOpenAIAnthropicGoogleAWSNVIDIAHugging FaceMicrosoftOllamaGroqFireworksIntegrations by componentCha

## Creating embbedings

In [ ]:
!pip install -qU langchain langchain-huggingface sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
!pip install -qU langchain-chroma

In [ ]:
!pip install "opentelemetry-api==1.38.0" "opentelemetry-sdk==1.38.0" "opentelemetry-exporter-otlp-proto-grpc==1.38.0" --quiet

In [ ]:
!pip install -qU langchain_community

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="WebDocs",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"
)

document_ids = vector_store.add_documents(documents=all_splits)
print(len(document_ids))

sample = vector_store.get(limit=1, include=["metadatas", "documents"])
print(sample)

58
{'ids': ['0ecc9611-2de9-4a8e-bc8b-5e0c8ab002a0'], 'embeddings': None, 'documents': ['Popular Providers\n- [OpenAI](/oss/python/integrations/providers/openai)\n- [Anthropic](/oss/python/integrations/providers/anthropic)\n- [Google](/oss/python/integrations/providers/google)\n- [AWS](/oss/python/integrations/providers/aws)\n- [NVIDIA](/oss/python/integrations/providers/nvidia)\n- [Hugging Face](/oss/python/integrations/providers/huggingface)\n- [Microsoft](/oss/python/integrations/providers/microsoft)\n- [Ollama](/oss/python/integrations/providers/ollama)\n- [Groq](/oss/python/integrations/providers/groq)\n- [Fireworks](/oss/python/integrations/providers/fireworks)'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'https://docs.langchain.com/oss/python/integrations/document_loaders'}]}


##retrive and generate

In [ ]:
def retrive_context(query: str, k: int = 6):
    retrived_docs = vector_store.similarity_search(query, k=k)
    docs_content = ""
    for doc in retrived_docs:
        docs_content += f"Source: {doc.metadata['source']}\n"
        docs_content += f"Content: {doc.page_content}\n\n"
    return docs_content, retrived_docs

## Model

In [ ]:
!pip install -qU langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get("gemini_api_key")
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key = api_key,
)


In [ ]:
def ask_about_pdf(user_query):
    context, docs = retrive_context(user_query, k=6)   # was k=2
    system_message = f""" You are a helpful AI assistant specialized in explaining LangChain documentation.
                          Use the following context extracted from the LangChain documentation webpages to answer the user's question:

  {context}

  INSTRUCTIONS:
  1. Answer based ONLY on the provided context from the LangChain documentation.
  2. Do NOT use outside knowledge beyond the documentation.
  3. If the answer is not present in the context, say:
     "This information is not available in the provided LangChain documentation context."
  4. Use exact terminology, class names, method names, and concepts as described.
  5. Keep explanations clear and technically accurate.
  6. Mention specific modules, classes, or configuration options when relevant.
  7. If code snippets are present, explain without modifying logic."""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_query}
    ]
    response = model.invoke(messages)
    return {
        "answer": response.content,
        "source_documents": docs,
        "context_used": context
    }

In [ ]:
from IPython.display import display, Markdown

In [ ]:
result = ask_about_pdf("How to use the HuggingFaceEmbeddings?")
display(Markdown(result["answer"]))

This information is not available in the provided LangChain documentation context. The context only lists "Hugging Face" as an embedding model integration and indicates there is a guide available, but it does not describe how to use `HuggingFaceEmbeddings`.

In [ ]:
result = ask_about_pdf("Explain the use this value : sentence-transformers/all-mpnet-base-v2")
display(Markdown(result["answer"]))

The value `all-mpnet-base-v2` is mentioned in the LangChain documentation as an example of a **classic embedding model** that has a context length cap of **512 tokens**.

In [ ]:
result = ask_about_pdf("How to use Open AI Embeddings")
display(Markdown(result["answer"]))

This information is not available in the provided LangChain documentation context. The context indicates that OpenAI is an embedding model integration and points to a "View guide," but it does not detail the steps on how to use OpenAI Embeddings.

In [ ]:
result = ask_about_pdf("Explain about this: OpenAIEmbeddings")
display(result["answer"])

"`OpenAIEmbeddings` is an `Embeddings` subclass used in LangChain for embedding model integrations.\n\nSpecifically, in the context of common deployment patterns, `OpenAIEmbeddings` is used in pattern (4) against TEI's OpenAI-compatible endpoint. OpenAI is also listed as a top integration for embedding models, with a guide available."